# 02 · Yakın-kopya doğrulama ve klasik baseline (HOG + Linear SVM)

1. pHash adaylarını SSIM ve korelasyonla piksel düzeyinde doğrular
2. HOG + doğrusal SVM baseline modelini eğitir; hiperparametreler **yalnızca doğrulama** kümesinde macro F1 ile seçilir

Test kümesine bu notebook'ta dokunulmaz. GPU gerekmez (yaklaşık 15–25 dk).

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')
# Kod ve çıktılar Drive'da tek klasörde durur; ham veri oturum diskinde (/content/data) tutulur.
PROJECT = next((p for p in ('/content/drive/MyDrive/YZProje', '/content/drive/MyDrive/YZProje/YZProje')
                if os.path.isfile(p + '/src/config.py')), None)
assert PROJECT, "Drive'da YZProje/src/config.py bulunamadı: proje klasörünü MyDrive/YZProje olarak yerleştirin."
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
os.environ['KNEE_OUTPUT_DIR'] = PROJECT + '/outputs'
os.environ['KNEE_DATA_ROOT'] = '/content/data'
print('Proje klasörü:', PROJECT)

In [ ]:
!pip install -q imagehash==4.3.2 tabulate==0.9.0

In [ ]:
# Görüntüler oturum diskinde yoksa: önce Drive'daki arşiv kopyası, o da yoksa bağlantıdan indirip arşivi Drive'a kaydeder.
from src import config as C, data_prep
print('Veri hazır:', data_prep.ensure_data())

## 1. pHash adaylarının doğrulanması

In [ ]:
from src import dup_verify
dup_verify.main([])
from IPython.display import Image, display
display(Image(filename=str(C.FIG_DIR / 'audit_duplicate_verification.png'), width=600))

## 2. HOG + Linear SVM

In [ ]:
from src import baseline
baseline.main([])

In [ ]:
import pandas as pd
g = pd.read_csv(C.OUTPUT_DIR / 'models/baseline_hog_svm/grid_search_val.csv')
cols = ['ppc','C','class_weight','n_features','val_macro_f1','val_balanced_accuracy','val_qwk','val_accuracy']
display(g[cols].sort_values('val_macro_f1', ascending=False).round(4))
display(pd.read_csv(C.OUTPUT_DIR / 'models/baseline_hog_svm/val_per_class.csv').round(3))
display(pd.read_csv(C.OUTPUT_DIR / 'results.csv').T)

Çıktıları (`outputs/audit/duplicate_verification.json`, `outputs/models/baseline_hog_svm/grid_search_val.csv`, `outputs/results.csv`) değerlendirme için paylaşın.